In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# Step 1: Define a large Conv3D model
class Conv3DModel(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_channels=32):
        super(Conv3DModel, self).__init__()
        self.conv1 = nn.Conv3d(in_channels, base_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv3d(base_channels, base_channels * 2, kernel_size=3, padding=1)
        self.conv3 = nn.Conv3d(base_channels * 2, base_channels * 4, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(base_channels * 4 * 16 * 16 * 16, 1024)
        self.fc2 = nn.Linear(1024, out_channels)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool3d(2)
        self.flatten = nn.Flatten()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.maxpool(x)
        x = self.relu(self.conv2(x))
        x = self.maxpool(x)
        x = self.relu(self.conv3(x))
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Step 2: Create a dummy 3D dataset (e.g., 3D volumes)
batch_size = 8  # Adjust batch size for testing
num_samples = 1000  # Number of dummy samples
in_channels = 1  # Grayscale or single-channel 3D data (e.g., CT scan)
out_channels = 1  # Regression task (single output value per sample)

# Generate random 3D volumes of size [batch_size, in_channels, depth, height, width]
dummy_data = torch.randn(num_samples, in_channels, 32, 32, 32)  # 32x32x32 volumes
dummy_labels = torch.randn(num_samples, out_channels)  # Random labels for regression

# Create a DataLoader
dataset = TensorDataset(dummy_data, dummy_labels)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Step 3: Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Conv3DModel(in_channels=in_channels, out_channels=out_channels)

# Step 4: Use DataParallel for multi-GPU training (if multiple GPUs are available)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model, device_ids=[0, 1, 2, 3])  # Modify device_ids for your GPU setup

model.to(device)

# Step 5: Define optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()  # Using MSELoss for regression task

# Step 6: Training loop
n_epochs = 10  # Set number of epochs

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        running_loss += loss.item()

        loss.backward()
        optimizer.step()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{n_epochs}], Loss: {avg_loss:.4f}")

print("Training completed")


Using 4 GPUs!


RuntimeError: Caught RuntimeError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/home/nikhil/g17p3/g17-p3/g15p3/lib/python3.11/site-packages/torch/nn/parallel/parallel_apply.py", line 85, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nikhil/g17p3/g17-p3/g15p3/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1518, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nikhil/g17p3/g17-p3/g15p3/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1527, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1120918/839278287.py", line 28, in forward
    x = self.relu(self.fc1(x))
                  ^^^^^^^^^^^
  File "/home/nikhil/g17p3/g17-p3/g15p3/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1518, in _wrapped_call_impl
    return 

: 